In [ ]:
%pip install /Workspace/Users/neil.braun@mirakl.com/.bundle/fast-gnn-benchmark/dev/files
dbutils.library.restartPython()

In [ ]:
import os
import boto3, json, io
import pyspark.sql.functions as F

In [ ]:
sessions_raw_val = spark.read.parquet("s3://mirakl-data-science-tmp2/nbraun/datasets/coview-mdm/sessions_raw_val.parquet")
prod_results_val = spark.read.parquet("s3://mirakl-data-science-tmp2/nbraun/datasets/coview-mdm/prod_results_val.parquet")
model_results_val = spark.read.parquet("s3://mirakl-data-science-tmp2/nbraun/datasets/coview-mdm/model_results_val.parquet")

print(f"sessions_raw_val: {sessions_raw_val.count()} triggers")
display(sessions_raw_val.limit(1))

print(f"prod_results_val: {prod_results_val.count()} triggers")
display(prod_results_val.limit(1))

print(f"model_results_val: {model_results_val.count()} triggers")
display(model_results_val.limit(1))

In [ ]:
product_metadata_val = (
    sessions_raw_val
    .select(F.explode("session_products").alias("product"))
    .select(F.col("product.internal_id").alias("internal_id"))
    .union(
        prod_results_val
        .select(F.explode("products_returned").alias("product"))
        .select(F.col("product.internal_id").alias("internal_id"))
    )
    .union(
        model_results_val
        .select(F.explode("products_returned").alias("product"))
        .select(F.col("product.internal_id").alias("internal_id"))
    )
    .union(
        sessions_raw_val
        .select(F.col("trigger_internal_id").alias("internal_id"))
    )
    .distinct()
)

display(product_metadata_val.limit(10))
display(product_metadata_val.count())

def get_customer_db_name(customer_shortname: str) -> str:
    df_customer = (
        spark.table("mirakl_ai.ds_etl_prod.t2s_gold_customer")
        .where(F.col("shortName") == customer_shortname)
        .select(F.col("databaseName").alias("db_name"))
    )
    return df_customer.collect()[0]["db_name"]


db_name = get_customer_db_name("maisons-du-monde")

df_products = (
    spark.table("mirakl_ai.ds_etl_prod.t2s_mongo_product_0_current")
    .filter(F.col("db_name") == db_name)
    .select(F.col("internalId").cast("bigint").alias("internal_id"), "name", "imageUrl")
    .dropDuplicates(["internal_id"])
)

product_metadata_val = (
    product_metadata_val
    .join(df_products, on="internal_id", how="left")
)

nb_unresolved = product_metadata_val.filter(F.col("name").isNull()).count()
print(f"produits non résolus dans le catalogue: {nb_unresolved}")

display(product_metadata_val.limit(10))

In [ ]:
product_metadata_val.write.mode("overwrite").parquet(
    "s3://mirakl-data-science-tmp2/nbraun/datasets/coview-mdm/product_metadata_val.parquet"
)
print("product_metadata_val.parquet uploaded")

In [ ]:
sessions_raw_val_size = (
    sessions_raw_val
    .select(
        F.col("exec_code"),
        F.col("trigger_internal_id"),
        F.size(
            F.filter(
                F.col("session_products"),
                lambda p: p["internal_id"] != F.col("trigger_internal_id")
            )
        ).alias("session_size"),
    )
)

display(sessions_raw_val_size.limit(1))

In [ ]:
full_results = (
    prod_results_val
    .select(
        F.col("exec_code"),
        F.col("trigger_internal_id"),
        F.col("positives").alias("positives_prod"),
        F.col("negatives").alias("negatives_prod"),
        F.col("products_returned").alias("products_returned_prod"),
    )
    .join(
        model_results_val.select(
            F.col("exec_code"),
            F.col("positives").alias("positives_model"),
            F.col("negatives").alias("negatives_model"),
            F.col("products_returned").alias("products_returned_model"),
        ),
        on="exec_code",
        how="left",
    )
    .join(
        sessions_raw_val_size.select("exec_code", "session_size"),
        on="exec_code",
        how="left",
    )
)

display(full_results.limit(1))

In [ ]:
full_results_with_metrics = (
    full_results
    .withColumns({
        "model_found": F.size("positives_model") > 0,
        "prod_found": F.size("positives_prod") > 0,
        "recall_model": F.when(
            F.col("session_size") > 0,
            F.size("positives_model") / F.col("session_size")
        ),
        "recall_prod": F.when(
            F.col("session_size") > 0,
            F.size("positives_prod") / F.col("session_size")
        ),
    })
)

display(full_results_with_metrics.limit(1))

In [ ]:
full_results_agg = (
    full_results_with_metrics.agg(
        F.avg("recall_model").alias("avg_recall_model"),
        F.avg("recall_prod").alias("avg_recall_prod"),
        F.avg(F.col("model_found").cast("int")).alias("pct_model_found"),
        F.avg(F.col("prod_found").cast("int")).alias("pct_prod_found"),
        F.avg((~F.col("model_found")).cast("int")).alias("pct_model_not_found"),
        F.avg((~F.col("prod_found")).cast("int")).alias("pct_prod_not_found"),
        F.count(F.when(F.col("recall_model").isNull(), 1)).alias("nb_excluded_recall_model"),
        F.count(F.when(F.col("recall_prod").isNull(), 1)).alias("nb_excluded_recall_prod"),
        F.count(F.when(F.col("model_found").isNull(), 1)).alias("nb_missing_model"),
        F.count(F.when(F.col("prod_found").isNull(), 1)).alias("nb_missing_prod"),
        F.count("*").alias("total_triggers"),
    )
)

display(full_results_agg)

In [ ]:
nb_excluded_matrix = full_results_with_metrics.filter(
    F.col("model_found").isNull() | F.col("prod_found").isNull()
).count()

full_results_with_metrics_no_null = full_results_with_metrics.filter(
    F.col("model_found").isNotNull() & F.col("prod_found").isNotNull()
)

total_valid = full_results_with_metrics_no_null.count()

matrix_2x2 = (
    full_results_with_metrics_no_null
    .groupBy("model_found")
    .pivot("prod_found", [True, False])
    .count()
    .fillna(0)
)

all_cases = spark.createDataFrame(
    [("both_find",), ("model_only",), ("prod_only",), ("neither",)],
    ["case"]
)

matrix_2x2_labeled = (
    all_cases
    .join(
        full_results_with_metrics_no_null
        .withColumn(
            "case",
            F.when(F.col("model_found") & F.col("prod_found"), "both_find")
            .when(F.col("model_found") & ~F.col("prod_found"), "model_only")
            .when(~F.col("model_found") & F.col("prod_found"), "prod_only")
            .otherwise("neither")
        )
        .groupBy("case")
        .count(),
        on="case",
        how="left"
    )
    .withColumn("count", F.coalesce(F.col("count"), F.lit(0)))
    .withColumn("pct", F.round(F.col("count") / total_valid * 100, 2))
)

print(f"triggers exclus (données manquantes d'un côté ou l'autre): {nb_excluded_matrix}")
print(f"triggers valides pour la matrice 2x2: {total_valid}")
display(matrix_2x2)
display(matrix_2x2_labeled.orderBy("case"))

In [ ]:
alignment_check = (
    sessions_raw_val.select("exec_code", F.col("trigger_internal_id").alias("tid_session"))
    .join(
        model_results_val.select("exec_code", F.col("trigger_internal_id").alias("tid_model")),
        on="exec_code",
        how="inner",
    )
    .withColumn("match", F.col("tid_session") == F.col("tid_model"))
    .groupBy("match")
    .count()
)

print(f"lignes comparées: {alignment_check.agg(F.sum('count')).first()[0]}")
display(alignment_check)

Statistics on full negatives sessions 

In [ ]:
prod_negative_only_results_val = spark.read.parquet("s3://mirakl-data-science-tmp2/nbraun/datasets/coview-mdm/prod_negative_only_results_val.parquet")
model_negative_only_results_val = spark.read.parquet("s3://mirakl-data-science-tmp2/nbraun/datasets/coview-mdm/model_negative_only_results_val.parquet")

print(f"prod_negative_only_results_val: {prod_negative_only_results_val.count()} triggers")
display(prod_negative_only_results_val.limit(1))

print(f"model_negative_only_results_val: {model_negative_only_results_val.count()} triggers")
display(model_negative_only_results_val.limit(1))

In [ ]:
negative_only_session_size = (
    sessions_raw_val_size
    .join(
        prod_negative_only_results_val.select("exec_code").distinct(),
        on="exec_code",
        how="left_semi",
    )
)

print(f"negative_only_session_size: {negative_only_session_size.count()} triggers")
display(negative_only_session_size.limit(1))

In [ ]:
full_results_negative_only = (
    prod_negative_only_results_val
    .select(
        F.col("exec_code"),
        F.col("trigger_internal_id"),
        F.col("positives").alias("positives_prod"),
        F.col("negatives").alias("negatives_prod"),
        F.col("products_returned").alias("products_returned_prod"),
    )
    .join(
        model_negative_only_results_val.select(
            F.col("exec_code"),
            F.col("positives").alias("positives_model"),
            F.col("negatives").alias("negatives_model"),
            F.col("products_returned").alias("products_returned_model"),
        ),
        on="exec_code",
        how="left",
    )
    .join(
        negative_only_session_size.select("exec_code", "session_size"),
        on="exec_code",
        how="left",
    )
)

display(full_results_negative_only.limit(1))

In [ ]:
full_results_negative_only_with_metrics = (
    full_results_negative_only
    .withColumns({
        "model_found": F.size("positives_model") > 0,
        "prod_found": F.size("positives_prod") > 0,
        "recall_model": F.when(
            F.col("session_size") > 0,
            F.size("positives_model") / F.col("session_size")
        ),
        "recall_prod": F.when(
            F.col("session_size") > 0,
            F.size("positives_prod") / F.col("session_size")
        ),
    })
)

display(full_results_negative_only_with_metrics.limit(1))

In [ ]:
full_results_negative_only_agg = (
    full_results_negative_only_with_metrics.agg(
        F.avg("recall_model").alias("avg_recall_model"),
        F.avg("recall_prod").alias("avg_recall_prod"),
        F.avg(F.col("model_found").cast("int")).alias("pct_model_found"),
        F.avg(F.col("prod_found").cast("int")).alias("pct_prod_found"),
        F.avg((~F.col("model_found")).cast("int")).alias("pct_model_not_found"),
        F.avg((~F.col("prod_found")).cast("int")).alias("pct_prod_not_found"),
        F.count(F.when(F.col("recall_model").isNull(), 1)).alias("nb_excluded_recall_model"),
        F.count(F.when(F.col("recall_prod").isNull(), 1)).alias("nb_excluded_recall_prod"),
        F.count(F.when(F.col("model_found").isNull(), 1)).alias("nb_missing_model"),
        F.count(F.when(F.col("prod_found").isNull(), 1)).alias("nb_missing_prod"),
        F.count("*").alias("total_triggers"),
    )
)

display(full_results_negative_only_agg)

In [ ]:
nb_excluded_matrix_negative_only = full_results_negative_only_with_metrics.filter(
    F.col("model_found").isNull() | F.col("prod_found").isNull()
).count()

full_results_negative_only_with_metrics_no_null = full_results_negative_only_with_metrics.filter(
    F.col("model_found").isNotNull() & F.col("prod_found").isNotNull()
)

total_valid_negative_only = full_results_negative_only_with_metrics_no_null.count()

matrix_2x2_negative_only = (
    full_results_negative_only_with_metrics_no_null
    .groupBy("model_found")
    .pivot("prod_found", [True, False])
    .count()
    .fillna(0)
)

matrix_2x2_negative_only_labeled = (
    all_cases
    .join(
        full_results_negative_only_with_metrics_no_null
        .withColumn(
            "case",
            F.when(F.col("model_found") & F.col("prod_found"), "both_find")
            .when(F.col("model_found") & ~F.col("prod_found"), "model_only")
            .when(~F.col("model_found") & F.col("prod_found"), "prod_only")
            .otherwise("neither")
        )
        .groupBy("case")
        .count(),
        on="case",
        how="left"
    )
    .withColumn("count", F.coalesce(F.col("count"), F.lit(0)))
    .withColumn("pct", F.round(F.col("count") / total_valid_negative_only * 100, 2))
)

print(f"triggers exclus (données manquantes d'un côté ou l'autre): {nb_excluded_matrix_negative_only}")
print(f"triggers valides pour la matrice 2x2: {total_valid_negative_only}")
display(matrix_2x2_negative_only)
display(matrix_2x2_negative_only_labeled.orderBy("case"))

In [ ]:
alignment_check_negative_only = (
    prod_negative_only_results_val.select("exec_code", F.col("trigger_internal_id").alias("tid_prod"))
    .join(
        model_negative_only_results_val.select("exec_code", F.col("trigger_internal_id").alias("tid_model")),
        on="exec_code",
        how="inner",
    )
    .withColumn("match", F.col("tid_prod") == F.col("tid_model"))
    .groupBy("match")
    .count()
)

print(f"lignes comparées: {alignment_check_negative_only.agg(F.sum('count')).first()[0]}")
display(alignment_check_negative_only)